# Text Summarization: Extractive & Abstractive

**Our task** is to build a text summarizer to summarize just one document of a single topic. We don't have any labels (like human-made summaries) to compare against and calculate metrics (which would've been ROUGE in that case). It therefore makes the output rather subjective, and the only objective qualitative measure we can present is the compression ratio - how much of the text have been reduced after summarization:

$$\text{compression\_rate}=\frac{len(\text{summary\_characters})}{len(\text{text\_characters})}$$

For the **data pre-processing**, we'll be using both `nltk` and `spaCy`, both of which do the same thing but both are used for practice.

**The baseline for the extractive summarization** is going to be `lead-3`, followed by TF-IDF and finally `TextRank`/`LexRank`. 

`lead-3` (essentially printing the first 3 sentences) is chosen because documents written with the point up front: news, reports, legal descriptions like this one, put the summary in the opening. That single passage contains the document's entire thesis. A frequency algorithm has to work hard to beat "just read the beginning."

TF-IDF, as the next simple but more complex logic, is comparable. We measure importance score per sentence based on the frequency of words appearing per sentence compared to the whole text, then sort and choose $30\%$ best of them. This method looks at a sentence in isolation and asks **"How many high-frequency words does this specific sentence have?"**.

`TextRank`/`LexRank` is the next level. They are based on Google's original PageRank algorithm, which ranked web pages based on how many other web pages linked to them. They look at the relationships between sentences and ask **"How much does this sentence look like all the other sentences in the document?"**. The algorithms create a square grid of size $N \times N$, where $N$ is the number of sentences. A sentence gets a high score if it is highly connected to other highly connected sentences. It represents the "consensus" of the text. 

The only difference is how each algorithm calculates the scores. `TextRank` uses the word overlap formula (how many common words two sentences share, normalized by their length), then the PageRank algorithm takes this static map (the matrix) and simulates a "random surfer" clicking from sentence to sentence based on those similarity weights. It multiplies the matrix by a vector over and over again until the scores stabilize and it figures out which sentences are the "hubs" that all the other sentences revolve around.

`TextRank`'s flaw is that it treats all words equally (sharing the word "orbiter" counts exactly the same as sharing the word "said"), `LexRank` fixes this by using TF-IDF and Cosine Similarity. Instead of just counting overlapping words, `LexRank` converts every sentence into a TF-IDF vector, it then calculates the exact geometric angle (Cosine Similarity) between those sentence vectors to build the $N \times N$ matrix.

For the **abstractive summarization**, we're going to use the middle ground. The (subjectively) best and concise output of the extractive summarization will be used as input for a transformer.

## Data preparation

The task calls for using both `SpaCy` and `nltk`, both of which do the same job — split into sentences, split into words. There is no reason to use both. We need to choose the more appropriate option based on the data, which, as mentioned, is a single document and a single topic.


| **Feature** | **NLTK** (2001, academic) | **spaCy** (2015, industrial) |
| :--- | :--- | :--- |
| **Design Philosophy**<br>(purpose) | A toolbox. We have to call separate functions for every single step. | A pipeline. We feed text in once, and it returns an object with everything calculated. |
| **Concept** | An unsupervised statistical model (`punkt`). It counts frequencies and looks for patterns in text. It builds a list of abbreviations (like _Mr._, _Dr._, or _Inc._) by calculating how likely a word is to appear before a period without ending a sentence.<br><br>Very fast, lightweight, requires very little computer memory, but doesn't understand the context or grammar. | A Deep Neural Network (CNN/transformer-based depending on the choice) parser that analyzes the grammatical structure of the entire text simultaneously. It builds a structural tree of the whole sentence. It figures out which words are subjects, verbs, and objects, and how they connect to each other; identifies sentence splits as a byproduct of understanding the overall syntax.<br><br>Highly accurate on complex, messy, or conversational text because it understands the full sentence context, not just punctuation rules, but computationally heavier, slower and requires more memory. |
| **Part of Speech (POS) Filtering** | Manual step. We must explicitly call `nltk.pos_tag()`. | Automatic. The `pos_` property (like `PROPN` or `VERB`) is ready instantly. |
| **Lemmatization**<br>(e.g., fly, flew, flown treated as a single entity) | Weak by default. Requires us to manually run a POS tagger first. | Strong and automatic. The `lemma_` property is immediately available for every word. |
| **Named Entities (NER)** | Clunky. Requires chaining several different tokenizers and chunkers. | Automatic. Entities like "NASA" or "1984" are immediately tagged in `doc.ents`. |
| **Best Use Case** | Academic learning or string manipulation. | Industrial applications, extracting deep meaning, and building actual products. |


Being simpler, lighter and faster, `nltk` might become our "data pre-processing baseline", but the concept of a baseline usually refers to something standard and param-default rather than a manually configurable drag. Even at this stage, it gives us no practical benefit. Unless we have to use heavy custom logic, I wouldn't go back to this option. We implement it only because I think it's a good show-case for learning.

### Data & constants

In [157]:
TEXT = "The Orbiter Discovery, OV-103, is considered eligible for listing in the National Register of Historic Places (NRHP) in the context of the U.S. Space Shuttle Program (1969-2011) under Criterion A in the areas of Space Exploration and Transportation and under Criterion C in the area of Engineering. Because it has achieved significance within the past fifty years, Criteria Consideration G applies. Under Criterion A, Discovery is significant as the oldest of the three extant orbiter vehicles constructed for the Space Shuttle Program (SSP), the longest running American space program to date; she was the third of five orbiters built by NASA. Unlike the Mercury, Gemini, and Apollo programs, the SSP’s emphasis was on cost effectiveness and reusability, and eventually the construction of a space station. Including her maiden voyage (launched August 30, 1984), Discovery flew to space thirty-nine times, more than any of the other four orbiters; she was also the first orbiter to fly twenty missions. She had the honor of being chosen as the Return to Flight vehicle after both the Challenger and Columbia accidents. Discovery was the first shuttle to fly with the redesigned SRBs, a result of the Challenger accident, and the first shuttle to fly with the Phase II and Block I SSME. Discovery also carried the Hubble Space Telescope to orbit and performed two of the five servicing missions to the observatory. She flew the first and last dedicated Department of Defense (DoD) missions, as well as the first unclassified defense-related mission. In addition, Discovery was vital to the construction of the International Space Station (ISS); she flew thirteen of the thirty-seven total missions flown to the station by a U.S. Space Shuttle. She was the first orbiter to dock to the ISS, and the first to perform an exchange of a resident crew. Under Criterion C, Discovery is significant as a feat of engineering. According to Wayne Hale, a flight director from Johnson Space Center, the Space Shuttle orbiter represents a “huge technological leap from expendable rockets and capsules to a reusable, winged, hypersonic, cargo-carrying spacecraft.” Although her base structure followed a conventional aircraft design, she used advanced materials that both minimized her weight for cargo-carrying purposes and featured low thermal expansion ratios, which provided a stable base for her Thermal Protection System (TPS) materials. The Space Shuttle orbiter also featured the first reusable TPS; all previous spaceflight vehicles had a single-use, ablative heat shield. Other notable engineering achievements of the orbiter included the first reusable orbital propulsion system, and the first two-fault-tolerant Integrated Avionics System. As Hale stated, the Space Shuttle remains “the largest, fastest, winged hypersonic aircraft in history,” having regularly flown at twenty-five times the speed of sound."
SUMMARIZATION_COEF = 0.3

In order to process text, we need to encode, and to encode it needs to be tokenized. The first question to ask is *what should one token be?*. We can tokenize into characters, words, morphemes. But **extractive summarization** itself is purely statistical. It counts frequencies and similarities giving the exact verbatim pieces, we have to use words to properly measure the similarities and count occurrences. That's why **extractive summarization** uses words as tokens.

Then, the next thing to address is **the language**. The text is in English, we want to divide it into words, but first into sentences, thi is called **sentence boundary detection (splitting)**. We can't just split it by periods (`.`) because there are proper nouns like `U.S.` or rely on capital layers for that reason; there are also abbreviations. `nltk` and `SpaCy` have pre-trained unsupervised ML models to split properly.

To consider how to handle **Proper Nouns and Named Entities** (like *Discovery* or *NASA*), as they carry the highest informational weight in a text, we will utilize **Part of Speech (POS) filtering**. By leveraging `nltk` and `SpaCy`'s grammar tagging, we can explicitly retain only Proper Nouns, Common Nouns, Verbs, and Adjectives. Additionally, `SpaCy`'s out-of-the-box **Named Entity Recognition (NER)** allows us to identify organizations, dates, and locations, which can optionally be used to give higher importance scores to sentences containing concrete facts.

There are also tenses and irregular verbs in the English language that are all supposed to be treated equally, we need lemmatization (supplied by `nltk` and `SpaCy`), it will treat `flying`, `flew`, `flown`, `fly`" as just `fly` correctly capturing the context. **Case normalization** (lowercasing) also contributes to the solution of this problem.

Typical high-frequency words called stop-words like `the`, `a`, `an`, `will`, `re`, `across`, `not` etc. have to be stripped as they don't express what the text is about not to appear in the top useful. `nltk` and `SpaCy` supply the ready-made lists.

Punctuation also doesn't help in **extractive summarization** and shouldn't become tokens, we will be removing them, the libraries also provide punctuation sets.

After splitting into sentences, "fitting" the lead-3 baseline becomes viable.

**For TF-IDF** strategy:

$$\text{tf-idf}(t, d) = \underbrace{\text{tf}(t,d)}_{\substack{\text{how often term } t \\ \text{appears in \emph{this} doc}}} \times \underbrace{\text{idf}(t)}_{\substack{\text{how \emph{rare} } t \text{ is} \\ \text{across all docs}}}$$

$$\text{tf}(t,d)=\frac{\text{Number of times term }t\text{ appears in document }d}{\text{Total number of terms in document }d}$$

$$\begin{gather*}
\text{idf}(t) = \ln{\left(\frac{1+N}{1+\text{df}(t)}\right)} + 1 \\ \\ 
	\begin{aligned} 
		&\bullet \text{ $N$ — total number of documents} \\
		&\bullet \text{ $\text{df}(t)$ — \emph{document frequency}: how many documents contain $t$ at all} \\
		&\bullet \text{ the $+1$s — smoothing, so a term in every document doesn't give $\ln(1)=0$ and vanish entirely} 
	\end{aligned} 
\end{gather*}$$

We only have the **TF** part because there is only one document. Usually, **TF-IDF** is used on documents and rank how well they fit certain query (search engines). And the reason the **IDF** part is there is because it measures how rare a term is across all the documents. Therefore, we just normalize be sentence length:

$$\begin{gather*}
\text{score}(\text{sentence}) = \frac{1}{\text{sent\_len}}\sum_{w \in \text{sent}} \text{freq}(w), \qquad \text{freq}(w) = \frac{\text{count}(w)}{\text{total\_count}(v)} \\ \\
\begin{aligned}
&\bullet \text{ count($w$) - word count within a sentence}  \\
&\bullet \text{ total\_count($v$) - word count across the whole vocabulary set (document)}  \\
\end{aligned}
\end{gather*}$$

Division by `max_count` is cosmetic - it puts scores on $[0,1]$ so they are readable.

### nltk pre-processing

In [ ]:
import string

import nltk
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import sent_tokenize, word_tokenize

In [ ]:
# NLTK ships without data - download once
"""
• stopwords - the list of uninformative words like articles, particles etc.
• wordnet - to convert NER labels
"""
for pkg in ['stopwords', 'wordnet']:
    nltk.download(pkg, quiet=True)

In [13]:
print(f"10 stop words examples: {stopwords.words('english')[:10]}")
print(f"Punctuation examples: {string.punctuation}")

10 stop words examples: ['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an']
Punctuation examples: !"#$%&'()*+,-./:;<=>?@[\]^_`{|}~


In [76]:
stop_words = set(stopwords.words("english"))
punctuation = set(string.punctuation) | {"''", "``", "--", "...", "'s", "’", "“", "”"}
lemmatizer = WordNetLemmatizer()

# split into sentences using the nltk's pre-trained unsupervised sentence tokenizer
nltk_raw_sentences = sent_tokenize(TEXT)
nltk_raw_sentences

['The Orbiter Discovery, OV-103, is considered eligible for listing in the National Register of Historic Places (NRHP) in the context of the U.S. Space Shuttle Program (1969-2011) under Criterion A in the areas of Space Exploration and Transportation and under Criterion C in the area of Engineering.',
 'Because it has achieved significance within the past fifty years, Criteria Consideration G applies.',
 'Under Criterion A, Discovery is significant as the oldest of the three extant orbiter vehicles constructed for the Space Shuttle Program (SSP), the longest running American space program to date; she was the third of five orbiters built by NASA.',
 'Unlike the Mercury, Gemini, and Apollo programs, the SSP’s emphasis was on cost effectiveness and reusability, and eventually the construction of a space station.',
 'Including her maiden voyage (launched August 30, 1984), Discovery flew to space thirty-nine times, more than any of the other four orbiters; she was also the first orbiter to

The problem is that by default `nltk`'s lemmatizer assumes each word is a noun so we need to run POS (part of speech) tagger on the original data to get the proper part of speech. However, because `nltk` is an old package of algorithms evolved over decades, not all of its methods agree directly. We can't just get the part of speech from `nltk`'s POS and pass to the lemmatizer because the `pos_tag` was trained on the Penn Treebank dataset and lemmatizer - on `wordnet`. We need to map the targets:

In [35]:
def nltk_to_wordnet_pos(nltk_tag: str):
    # NOTE: pos_tag has way more parts of speech, but we don't need such detailed distinction
    return {"J": wordnet.ADJ, "V": wordnet.VERB,
            "N": wordnet.NOUN, "R": wordnet.ADV}.get(nltk_tag[0], wordnet.NOUN)

In [142]:
def nltk_tokenize(text):
    # split into unique words considering lemmatization, stop words and NER filtering
    words = []
    word_candidates = word_tokenize(text) # includes raw words, prepositions, punctuation etc.
    tagged_words = nltk.pos_tag(word_candidates) # this returns pairs like: [('Discovery', 'NNP'), ('flew', 'VBD')]
    for raw_w, nltk_tag in tagged_words:
        raw_w_lower = raw_w.lower()

        # if a word is a stop word or punctuation, skip it
        if (raw_w_lower in stop_words) or (raw_w_lower in punctuation):
            continue

        words.append(lemmatizer.lemmatize(raw_w_lower, nltk_to_wordnet_pos(nltk_tag)))
    return words

nltk_words = nltk_tokenize(TEXT)
print(f"Tokens: {nltk_words}")

Tokens: ['orbiter', 'discovery', 'ov-103', 'consider', 'eligible', 'list', 'national', 'register', 'historic', 'place', 'nrhp', 'context', 'u.s.', 'space', 'shuttle', 'program', '1969-2011', 'criterion', 'area', 'space', 'exploration', 'transportation', 'criterion', 'c', 'area', 'engineering', 'achieve', 'significance', 'past', 'year', 'criterion', 'consideration', 'g', 'applies', 'criterion', 'discovery', 'significant', 'old', 'extant', 'orbiter', 'vehicle', 'construct', 'space', 'shuttle', 'program', 'ssp', 'long', 'running', 'american', 'space', 'program', 'date', 'orbiter', 'build', 'nasa', 'unlike', 'mercury', 'gemini', 'apollo', 'program', 'ssp', 's', 'emphasis', 'cost', 'effectiveness', 'reusability', 'eventually', 'construction', 'space', 'station', 'include', 'maiden', 'voyage', 'launch', 'august', '30', '1984', 'discovery', 'fly', 'space', 'thirty-nine', 'time', 'orbiter', 'orbiter', 'fly', 'mission', 'honor', 'choose', 'return', 'flight', 'vehicle', 'challenger', 'columbia',

Successfully tokenized.

### SpaCy pre-processing

In [ ]:
import spacy

Unlike `nltk`, `SpaCy` is an automated pipeline that features different deep NN models. The first step is to choose the appropriate model for the task.

|         | size    | based on               | word vectors (embeddings)                       | NER F     | sentence F (splitting sentences)|
| ------- | ------- | ---------------------- | ---------------------------------- | --------- | ---------- |
| **sm**  | 14.5 MB | CNN (tok2vec)          | **none**                           | 0.843     | **0.907**  |
| **md**  | ~40 MB  | CNN, same architecture | 20,000 unique 300-dimension vectors | 0.843     | 0.907      |
| **lg**| ~560 MB | CNN, same architecture | ~500,000 unique 300-dimension vectors             | 0.844     | 0.907      |
| **trf** | ~440 MB | **RoBERTa-base**       | none                               | **0.901** | 0.897      |

`sm`, `md` and `lg` have essentially identical accuracy. `SpaCy`'s own maintainers published accuracy tables for the three are the same except NER recall. They do not differ by quality. They differ by **whether they ship word vectors**:
- Need `token.vector`, `doc.similarity()`, semantic comparison → **md or lg**
- Need only tokenization, POS, lemma, NER, sentence splits → **sm for identical result + cheaper**

`trf` (transformer-based, most accurate, slowest) is the only real accuracy jump, and only for NER (0.843 → 0.901) and parsing.

In [82]:
!python -m spacy download en_core_web_sm 

     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---- ----------------------------------- 1.3/12.8 MB 9.5 MB/s eta 0:00:02
     --------- ------------------------------ 2.9/12.8 MB 9.3 MB/s eta 0:00:02
     ------------------- -------------------- 6.3/12.8 MB 12.1 MB/s eta 0:00:01
     ------------------------------ -------- 10.0/12.8 MB 13.5 MB/s eta 0:00:01
     --------------------------------------  12.6/12.8 MB 13.8 MB/s eta 0:00:01
     --------------------------------------- 12.8/12.8 MB 13.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [102]:
# the splitting model
nlp = spacy.load("en_core_web_sm")

# split into sentences using the spacy's pre-trained NN
spacy_raw_text = nlp(TEXT)
spacy_raw_sentences = list(spacy_raw_text.sents)      # all document sentences, IN DOCUMENT ORDER
spacy_raw_sentences

[The Orbiter Discovery, OV-103, is considered eligible for listing in the National Register of Historic Places (NRHP) in the context of the U.S. Space Shuttle Program (1969-2011) under Criterion A in the areas of Space Exploration and Transportation and under Criterion C in the area of Engineering.,
 Because it has achieved significance within the past fifty years, Criteria Consideration G applies.,
 Under Criterion A, Discovery is significant as the oldest of the three extant orbiter vehicles constructed for the Space Shuttle Program (SSP), the longest running American space program to date; she was the third of five orbiters built by NASA.,
 Unlike the Mercury, Gemini, and Apollo programs, the SSP’s emphasis was on cost effectiveness and reusability, and eventually the construction of a space station.,
 Including her maiden voyage (launched August 30, 1984), Discovery flew to space thirty-nine times, more than any of the other four orbiters; she was also the first orbiter to fly twen

In [119]:
# each sentence in spacy is "spacy.tokens.span.Span" object and in nltk they are just str
spacy_raw_sentences_str = [str(sent) for sent in spacy_raw_sentences]
set(spacy_raw_sentences_str) == set(nltk_raw_sentences)

True

The split is identical

In [ ]:
# spacy "sm" model's stop words
stop_words = nlp.Defaults.stop_words

print(f"Stop words examples: {stop_words}")
print(f"Punctuation examples: {punctuation}")

Stop words examples: {'even', 'became', 'see', 'this', 'ever', 'these', 'five', 'latterly', 'or', 'next', 'before', 'me', 'seeming', 'enough', 'she', 'sometime', 'were', 'anyway', "'m", 'eleven', 'he', '‘ll', 'such', 'along', 'doing', 'around', '’re', 'might', 'we', 'does', 'nowhere', 'among', 'formerly', 'throughout', 'whatever', 'without', 'nor', 'should', 'twenty', 'hers', 'herein', 'own', 'beforehand', 'else', 'everything', 'six', 'onto', 'within', 'very', 'i', 'are', 'somewhere', 'yet', 'they', 'fifty', 'your', '’d', 'still', 'my', 'few', 'already', 'please', 'really', 'thus', 'eight', 'itself', 'whereafter', 'first', 'hence', 'thereafter', 'out', 'whoever', 'fifteen', 'although', 'whereby', 'sixty', 'myself', 'forty', 'however', 'towards', 'upon', 'otherwise', 'not', 'used', 'noone', '‘re', 'every', 'beside', 'into', 'may', 'cannot', 'ours', 'serious', 'show', 'nevertheless', '’ve', 'its', 'twelve', 'be', 'everyone', 'any', '‘m', 'up', 'front', 'top', 'being', 'hereby', 'often', 

In [151]:
def spacy_tokenize(text):
    words = []
    for token in nlp(str(text)):
        if token.is_stop or token.is_punct:
            continue
        words.append(token.lemma_.lower())
    return words

spacy_words = spacy_tokenize(TEXT)
print(f"Tokens: {spacy_words}")

Tokens: ['orbiter', 'discovery', 'ov-103', 'consider', 'eligible', 'list', 'national', 'register', 'historic', 'places', 'nrhp', 'context', 'u.s.', 'space', 'shuttle', 'program', '1969', '2011', 'criterion', 'area', 'space', 'exploration', 'transportation', 'criterion', 'c', 'area', 'engineering', 'achieve', 'significance', 'past', 'year', 'criteria', 'consideration', 'g', 'apply', 'criterion', 'discovery', 'significant', 'old', 'extant', 'orbiter', 'vehicle', 'construct', 'space', 'shuttle', 'program', 'ssp', 'long', 'running', 'american', 'space', 'program', 'date', 'orbiter', 'build', 'nasa', 'unlike', 'mercury', 'gemini', 'apollo', 'program', 'ssp', 'emphasis', 'cost', 'effectiveness', 'reusability', 'eventually', 'construction', 'space', 'station', 'include', 'maiden', 'voyage', 'launch', 'august', '30', '1984', 'discovery', 'fly', 'space', 'thirty', 'time', 'orbiter', 'orbiter', 'fly', 'mission', 'honor', 'choose', 'return', 'flight', 'vehicle', 'challenger', 'columbia', 'acciden

## Extractive Summarization

### Baseline

In [ ]:
def lead_3_baseline(raw_sentences):
    return " ".join(raw_sentences[:3])

#### `nltk`'s lead-3

In [77]:
nltk_lead_3_summary = lead_3_baseline(nltk_raw_sentences)
nltk_lead_3_summary

'The Orbiter Discovery, OV-103, is considered eligible for listing in the National Register of Historic Places (NRHP) in the context of the U.S. Space Shuttle Program (1969-2011) under Criterion A in the areas of Space Exploration and Transportation and under Criterion C in the area of Engineering. Because it has achieved significance within the past fifty years, Criteria Consideration G applies. Under Criterion A, Discovery is significant as the oldest of the three extant orbiter vehicles constructed for the Space Shuttle Program (SSP), the longest running American space program to date; she was the third of five orbiters built by NASA.'

#### `SpaCy`'s lead-3

In [120]:
spacy_lead_3_summary = lead_3_baseline(spacy_raw_sentences_str)
spacy_lead_3_summary

'The Orbiter Discovery, OV-103, is considered eligible for listing in the National Register of Historic Places (NRHP) in the context of the U.S. Space Shuttle Program (1969-2011) under Criterion A in the areas of Space Exploration and Transportation and under Criterion C in the area of Engineering. Because it has achieved significance within the past fifty years, Criteria Consideration G applies. Under Criterion A, Discovery is significant as the oldest of the three extant orbiter vehicles constructed for the Space Shuttle Program (SSP), the longest running American space program to date; she was the third of five orbiters built by NASA.'

### TF-IDF

#### nltk

In [ ]:
def tf_frequencies(document_tokens):
    frequencies = {}
    for word in document_tokens:
        # Get current count (default to 0 if not found) and add 1
        frequencies[word] = frequencies.get(word, 0) + 1

    # normalization by total words
    frequencies = {k:v/len(document_tokens) for k,v in frequencies.items()}
    return frequencies

nltk_frequencies = tf_frequencies(nltk_words)
sorted(nltk_frequencies.items(), key=lambda item: item[1], reverse=True)

[('space', 0.052845528455284556),
 ('orbiter', 0.036585365853658534),
 ('shuttle', 0.032520325203252036),
 ('fly', 0.032520325203252036),
 ('discovery', 0.028455284552845527),
 ('criterion', 0.02032520325203252),
 ('mission', 0.02032520325203252),
 ('program', 0.016260162601626018),
 ('engineering', 0.012195121951219513),
 ('vehicle', 0.012195121951219513),
 ('station', 0.012195121951219513),
 ('reusable', 0.012195121951219513),
 ('system', 0.012195121951219513),
 ('u.s.', 0.008130081300813009),
 ('area', 0.008130081300813009),
 ('significant', 0.008130081300813009),
 ('ssp', 0.008130081300813009),
 ('construction', 0.008130081300813009),
 ('include', 0.008130081300813009),
 ('time', 0.008130081300813009),
 ('flight', 0.008130081300813009),
 ('challenger', 0.008130081300813009),
 ('accident', 0.008130081300813009),
 ('perform', 0.008130081300813009),
 ('iss', 0.008130081300813009),
 ('hale', 0.008130081300813009),
 ('wing', 0.008130081300813009),
 ('hypersonic', 0.008130081300813009),


In [144]:
def tf_sentence_scores(raw_sentences, tokenizer, tf_frequencies):
    sentence_scores = {}

    for i, sent in enumerate(raw_sentences):
        sentence_scores[i] = 0
        sent_tokens = tokenizer(str(sent))

        # If raw_sentences contains a short sentence made entirely of stop-words or punctuation (e.g., "And so on."),
        if not sent_tokens:
            continue

        sentence_tokens_counts = {}
        for token in sent_tokens:
            sentence_tokens_counts[token] = sentence_tokens_counts.get(token, 0) + 1

        for word, count in sentence_tokens_counts.items():
            sentence_scores[i] += count * tf_frequencies.get(word, 0)

        sentence_scores[i] = sentence_scores[i] / len(sent_tokens)

    return sentence_scores

In [161]:
nltk_sentence_scores = tf_sentence_scores(nltk_raw_sentences, nltk_tokenize, nltk_frequencies)
nltk_sentence_scores_sorted = sorted(nltk_sentence_scores.items(), key=lambda item: item[1], reverse=True)
nltk_sentence_scores_sorted

[(9, 0.01806684733514002),
 (4, 0.017784552845528455),
 (2, 0.01703445605884631),
 (6, 0.014518002322880369),
 (14, 0.014383989993746089),
 (0, 0.013445903689806128),
 (7, 0.013414634146341461),
 (11, 0.012195121951219514),
 (12, 0.012195121951219511),
 (16, 0.011065943992773258),
 (8, 0.01084010840108401),
 (10, 0.009872241579558653),
 (15, 0.009380863039399622),
 (3, 0.008943089430894311),
 (5, 0.0066056910569105695),
 (13, 0.006016260162601626),
 (1, 0.005589430894308944)]

In [178]:
def compose_summary_tf(raw_sentences: list[str], scored_sentences: list[tuple[int, float]]):
    # --- Select a fraction (30%) or at least 1 sentence ---
    select_length = max(1, int(len(raw_sentences) * SUMMARIZATION_COEF))

    top_30p_sentences = scored_sentences[:select_length]
    top_30p_indices = [i for i, sc in top_30p_sentences]

    # --- Restore DOCUMENT order ---
    # Because our items are just integer indices, restoring chronological order just means sorting
    # the numbers from lowest to highest.
    top_indices_sorted = sorted(top_30p_indices)
    
    # --- Build the summary ---
    return " ".join(str(raw_sentences[i]).strip() for i in top_indices_sorted)

In [179]:
def compression_rate(original_text, summarized_text):
    return len(summarized_text) / len(original_text)

In [180]:
nltk_summary_tf = compose_summary_tf(nltk_raw_sentences, nltk_sentence_scores_sorted)
nltk_summary_tf

'Under Criterion A, Discovery is significant as the oldest of the three extant orbiter vehicles constructed for the Space Shuttle Program (SSP), the longest running American space program to date; she was the third of five orbiters built by NASA. Including her maiden voyage (launched August 30, 1984), Discovery flew to space thirty-nine times, more than any of the other four orbiters; she was also the first orbiter to fly twenty missions. Discovery was the first shuttle to fly with the redesigned SRBs, a result of the Challenger accident, and the first shuttle to fly with the Phase II and Block I SSME. In addition, Discovery was vital to the construction of the International Space Station (ISS); she flew thirteen of the thirty-seven total missions flown to the station by a U.S. Space Shuttle. The Space Shuttle orbiter also featured the first reusable TPS; all previous spaceflight vehicles had a single-use, ablative heat shield.'

In [181]:
compression_rate(TEXT, nltk_summary_tf)

0.3234686854783207

#### SpaCy

In [152]:
spacy_frequencies = tf_frequencies(spacy_words)
sorted(spacy_frequencies.items(), key=lambda item: item[1], reverse=True)

[('space', 0.050980392156862744),
 ('orbiter', 0.03529411764705882),
 ('shuttle', 0.03137254901960784),
 ('fly', 0.03137254901960784),
 ('discovery', 0.027450980392156862),
 ('mission', 0.0196078431372549),
 ('program', 0.01568627450980392),
 ('criterion', 0.01568627450980392),
 ('engineering', 0.011764705882352941),
 ('vehicle', 0.011764705882352941),
 ('station', 0.011764705882352941),
 ('carry', 0.011764705882352941),
 ('reusable', 0.011764705882352941),
 ('system', 0.011764705882352941),
 ('u.s.', 0.00784313725490196),
 ('area', 0.00784313725490196),
 ('c', 0.00784313725490196),
 ('significant', 0.00784313725490196),
 ('ssp', 0.00784313725490196),
 ('construction', 0.00784313725490196),
 ('include', 0.00784313725490196),
 ('thirty', 0.00784313725490196),
 ('time', 0.00784313725490196),
 ('flight', 0.00784313725490196),
 ('challenger', 0.00784313725490196),
 ('accident', 0.00784313725490196),
 ('perform', 0.00784313725490196),
 ('defense', 0.00784313725490196),
 ('iss', 0.0078431372

In [160]:
spacy_sentence_scores = tf_sentence_scores(spacy_raw_sentences, spacy_tokenize, spacy_frequencies)
spacy_sentence_scores_sorted = sorted(spacy_sentence_scores.items(), key=lambda item: item[1], reverse=True)
spacy_sentence_scores_sorted

[(4, 0.017401960784313723),
 (9, 0.016924664602683177),
 (2, 0.01624649859943977),
 (6, 0.014005602240896352),
 (7, 0.013725490196078433),
 (14, 0.013165266106442574),
 (0, 0.012636165577342047),
 (11, 0.01241830065359477),
 (12, 0.011764705882352934),
 (16, 0.011072664359861588),
 (8, 0.010588235294117647),
 (10, 0.009523809523809523),
 (3, 0.009243697478991595),
 (15, 0.008683473389355743),
 (5, 0.006372549019607843),
 (13, 0.006033182503770738),
 (1, 0.00392156862745098)]

In [176]:
spacy_summary_tf = compose_summary_tf(spacy_raw_sentences, spacy_sentence_scores_sorted)
spacy_summary_tf

'Under Criterion A, Discovery is significant as the oldest of the three extant orbiter vehicles constructed for the Space Shuttle Program (SSP), the longest running American space program to date; she was the third of five orbiters built by NASA. Including her maiden voyage (launched August 30, 1984), Discovery flew to space thirty-nine times, more than any of the other four orbiters; she was also the first orbiter to fly twenty missions. Discovery was the first shuttle to fly with the redesigned SRBs, a result of the Challenger accident, and the first shuttle to fly with the Phase II and Block I SSME. Discovery also carried the Hubble Space Telescope to orbit and performed two of the five servicing missions to the observatory. In addition, Discovery was vital to the construction of the International Space Station (ISS); she flew thirteen of the thirty-seven total missions flown to the station by a U.S. Space Shuttle.'

In [177]:
compression_rate(TEXT, spacy_summary_tf)

0.32002752924982797

### TextRank

In [194]:
import math

import numpy as np
import networkx as nx

def build_textrank_matrix(tokenized_sentences):
    """
    Builds an N x N similarity matrix based on word overlap.
    tokenized_sentences: A list of lists containing cleaned, lemmatized words for each sentence.
    """
    N = len(tokenized_sentences)
    similarity_matrix = np.zeros((N, N))
    counter=0
    for i in range(N):
        # Start j from i + 1 to only calculate the upper triangle
        for j in range(i+1, N):
            # A sentence shouldn't vote for itself
            if i == j:
                continue 
            counter+=1
            # Convert to sets to get unique words
            words_i = set(tokenized_sentences[i])
            words_j = set(tokenized_sentences[j])
            
            # |S_i ∩ S_j| - how many words overlapped in the 2 sentences
            overlap = len(words_i.intersection(words_j))
            
            if overlap == 0:
                similarity_matrix[i][j] = 0
                continue
                
            # log(|S_i|) + log(|S_j|)
            log_len_i = math.log(len(words_i)) if len(words_i) > 1 else 0
            log_len_j = math.log(len(words_j)) if len(words_j) > 1 else 0
            
            denominator = log_len_i + log_len_j
            
            # Prevent zero division if both sentences have 1 or 0 words
            # AND # Mirror the score to both sides of the diagonal
            if denominator == 0:
                similarity_matrix[i][j] = 0
                similarity_matrix[j][i] = 0
            else:
                score = overlap / denominator
                similarity_matrix[i][j] = score
                similarity_matrix[j][i] = score

    return similarity_matrix

In [ ]:
def text_rank_scores(tokenized_sentences):
    # Build the N x N matrix and rank every pair of different sentences
    matrix = build_textrank_matrix(tokenized_sentences)

    # Convert the numpy matrix into a NetworkX graph
    similarity_graph = nx.from_numpy_array(matrix)

    # Run PageRank to get the final scores
    # This returns a dictionary formatted exactly like our TF scores: {0: 0.12, 1: 0.08, 2: 0.15...}
    textrank_scores = nx.pagerank(similarity_graph)

    # Sort the scores like before
    return sorted(textrank_scores.items(), key=lambda item: item[1], reverse=True)

#### nltk

In [201]:
# tokenize into words but keep the sentences separated
# list[ [sent_1_word_1, sent_1_word_2...], [sent_2_word_1, sent_2_word_2...] ] 
nltk_tokenized_sentences = [nltk_tokenize(sent) for sent in nltk_raw_sentences]

nltk_textrank_scores_sorted = text_rank_scores(nltk_tokenized_sentences)
nltk_textrank_scores_sorted

[(2, 0.10364458177320236),
 (0, 0.09961616095183952),
 (9, 0.09321425035316966),
 (4, 0.08944408071463922),
 (14, 0.0821138231385975),
 (12, 0.07783765802790768),
 (16, 0.06786491551259975),
 (7, 0.06154571030473241),
 (6, 0.06150334709597172),
 (11, 0.0517431648937389),
 (15, 0.04347804240135618),
 (3, 0.041584869131221586),
 (10, 0.033246639431995136),
 (8, 0.029691116097859765),
 (5, 0.023883366028250634),
 (13, 0.02139522672999592),
 (1, 0.018193047412921735)]

In [202]:
# Generate summary using the exact same function we wrote earlier
nltk_textrank_summary = compose_summary_tf(nltk_raw_sentences, nltk_textrank_scores_sorted)
print(nltk_textrank_summary)

The Orbiter Discovery, OV-103, is considered eligible for listing in the National Register of Historic Places (NRHP) in the context of the U.S. Space Shuttle Program (1969-2011) under Criterion A in the areas of Space Exploration and Transportation and under Criterion C in the area of Engineering. Under Criterion A, Discovery is significant as the oldest of the three extant orbiter vehicles constructed for the Space Shuttle Program (SSP), the longest running American space program to date; she was the third of five orbiters built by NASA. Including her maiden voyage (launched August 30, 1984), Discovery flew to space thirty-nine times, more than any of the other four orbiters; she was also the first orbiter to fly twenty missions. In addition, Discovery was vital to the construction of the International Space Station (ISS); she flew thirteen of the thirty-seven total missions flown to the station by a U.S. Space Shuttle. The Space Shuttle orbiter also featured the first reusable TPS; a

#### SpaCy

In [203]:
spacy_tokenized_sentences = [spacy_tokenize(sent) for sent in nltk_raw_sentences]
spacy_textrank_scores_sorted = text_rank_scores(spacy_tokenized_sentences)
spacy_textrank_scores_sorted

[(2, 0.09925180847698599),
 (9, 0.09558500610059882),
 (0, 0.09469345699253187),
 (4, 0.09252995082159515),
 (12, 0.08320313119732378),
 (14, 0.08085919511154638),
 (16, 0.0685686817626695),
 (7, 0.0681475649461319),
 (6, 0.06209097701156618),
 (11, 0.045821778880465604),
 (15, 0.04283976537026399),
 (3, 0.04231420477981301),
 (10, 0.033562117087088776),
 (8, 0.030251734828256084),
 (13, 0.026638984162678965),
 (5, 0.024353716771954415),
 (1, 0.009287925698529414)]

In [204]:
spacy_textrank_summary = compose_summary_tf(nltk_raw_sentences, spacy_textrank_scores_sorted)
print(spacy_textrank_summary)

The Orbiter Discovery, OV-103, is considered eligible for listing in the National Register of Historic Places (NRHP) in the context of the U.S. Space Shuttle Program (1969-2011) under Criterion A in the areas of Space Exploration and Transportation and under Criterion C in the area of Engineering. Under Criterion A, Discovery is significant as the oldest of the three extant orbiter vehicles constructed for the Space Shuttle Program (SSP), the longest running American space program to date; she was the third of five orbiters built by NASA. Including her maiden voyage (launched August 30, 1984), Discovery flew to space thirty-nine times, more than any of the other four orbiters; she was also the first orbiter to fly twenty missions. In addition, Discovery was vital to the construction of the International Space Station (ISS); she flew thirteen of the thirty-seven total missions flown to the station by a U.S. Space Shuttle. According to Wayne Hale, a flight director from Johnson Space Cen

### LexRank

In [205]:
import networkx as nx
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


def build_lexrank_scores(tokenized_sentences):
    # 1. sklearn expects strings, so we join our cleaned tokens back into sentences
    # e.g., ['orbiter', 'discovery', 'fly'] -> "orbiter discovery fly"
    cleaned_sentences = [" ".join(tokens) for tokens in tokenized_sentences]
    
    # 2. Automatically calculate TF-IDF for every word in every sentence
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(cleaned_sentences)
    
    # 3. Build the N x N similarity matrix using Cosine Similarity
    # (instead of the entire manual overlap loop)
    lexrank_matrix = cosine_similarity(tfidf_matrix)
    
    # 4. Run the same PageRank engine
    lexrank_graph = nx.from_numpy_array(lexrank_matrix)
    lexrank_scores = nx.pagerank(lexrank_graph)
    
    return lexrank_scores

#### nltk

In [207]:
# Get scores and sort them
nltk_lexrank_scores_dict = build_lexrank_scores(nltk_tokenize(sent) for sent in nltk_raw_sentences)
nltk_lexrank_scores_sorted = sorted(nltk_lexrank_scores_dict.items(), key=lambda item: item[1], reverse=True)
nltk_lexrank_scores_sorted

[(2, 0.07570210583012583),
 (9, 0.07460827157549724),
 (4, 0.07273368981468022),
 (0, 0.06601118661292035),
 (6, 0.06304417054159923),
 (12, 0.061939805464363944),
 (14, 0.06118962353681281),
 (11, 0.05932940298853309),
 (7, 0.05730301097717153),
 (16, 0.05644143725217661),
 (15, 0.05270817011537738),
 (3, 0.051206491076815186),
 (8, 0.050601498217266894),
 (10, 0.05037832324094266),
 (13, 0.049342612388391466),
 (5, 0.049098579719294315),
 (1, 0.04836162064803149)]

In [208]:
# Generate the summary using your existing function
nltk_lexrank_summary = compose_summary_tf(nltk_raw_sentences, nltk_lexrank_scores_sorted)
print(nltk_lexrank_summary)

The Orbiter Discovery, OV-103, is considered eligible for listing in the National Register of Historic Places (NRHP) in the context of the U.S. Space Shuttle Program (1969-2011) under Criterion A in the areas of Space Exploration and Transportation and under Criterion C in the area of Engineering. Under Criterion A, Discovery is significant as the oldest of the three extant orbiter vehicles constructed for the Space Shuttle Program (SSP), the longest running American space program to date; she was the third of five orbiters built by NASA. Including her maiden voyage (launched August 30, 1984), Discovery flew to space thirty-nine times, more than any of the other four orbiters; she was also the first orbiter to fly twenty missions. Discovery was the first shuttle to fly with the redesigned SRBs, a result of the Challenger accident, and the first shuttle to fly with the Phase II and Block I SSME. In addition, Discovery was vital to the construction of the International Space Station (ISS

#### SpaCy

In [209]:
# Get scores and sort them
spacy_lexrank_scores_dict = build_lexrank_scores(spacy_tokenize(sent) for sent in spacy_raw_sentences)
spacy_lexrank_scores_sorted = sorted(spacy_lexrank_scores_dict.items(), key=lambda item: item[1], reverse=True)
spacy_lexrank_scores_sorted

[(9, 0.07391330757004046),
 (2, 0.0734710867483213),
 (4, 0.073344966357063),
 (0, 0.06288478729810604),
 (6, 0.06246239866320415),
 (12, 0.062420677477960006),
 (14, 0.06040937826809412),
 (7, 0.05952382367114655),
 (1, 0.058823529411764705),
 (16, 0.05688344448341011),
 (11, 0.055295338349146184),
 (15, 0.05230289294942706),
 (3, 0.05059289491851291),
 (8, 0.05017446774792348),
 (10, 0.04986316235050852),
 (13, 0.04901005489001992),
 (5, 0.0486237888453515)]

In [210]:
# Generate the summary using your existing function
spacy_lexrank_summary = compose_summary_tf(spacy_raw_sentences, spacy_lexrank_scores_sorted)
print(spacy_lexrank_summary)

The Orbiter Discovery, OV-103, is considered eligible for listing in the National Register of Historic Places (NRHP) in the context of the U.S. Space Shuttle Program (1969-2011) under Criterion A in the areas of Space Exploration and Transportation and under Criterion C in the area of Engineering. Under Criterion A, Discovery is significant as the oldest of the three extant orbiter vehicles constructed for the Space Shuttle Program (SSP), the longest running American space program to date; she was the third of five orbiters built by NASA. Including her maiden voyage (launched August 30, 1984), Discovery flew to space thirty-nine times, more than any of the other four orbiters; she was also the first orbiter to fly twenty missions. Discovery was the first shuttle to fly with the redesigned SRBs, a result of the Challenger accident, and the first shuttle to fly with the Phase II and Block I SSME. In addition, Discovery was vital to the construction of the International Space Station (ISS

## Abstractive summarization

In [ ]:
from transformers import pipeline

# stage 1 - your existing extractive summarizer, but keep MORE than final
extracted, _, _ = summarize_nltk(TEXT, ratio=0.5)     # ~50% instead of 30%

# stage 2 - abstractive rewrite
summarizer = pipeline("summarization", model="sshleifer/distilbart-cnn-12-6")
result = summarizer(extracted, max_length=120, min_length=40, do_sample=False)
print(result[0]["summary_text"])